Rota Maker

Times:
- Open Start = 8
- Close Finish = 11
- Busy Hours = 12-2, 5-7
- Quiet Hours = 3-4
- Shift Size = 4-8hrs

In [17]:
from dataclasses import dataclass, field
from datetime import date, time, datetime, timedelta

In [18]:
from typing import Required
class Employee:
  _id_counter = 1 #Class variable

  def __init__(self, name, contract_hours, max_shifts_per_week,
               max_hours_per_day, job_role):
    self.id = Employee._id_counter #Make a unique ID
    Employee._id_counter += 1 #Increase counter
    self.name = name #Who is it
    self.contract_hours = contract_hours #Min hours
    self.assigned_hours = 0 #How many hours are they working
    self.max_shifts_per_week = max_shifts_per_week #How many days are they in
    self.max_hours_per_day = max_hours_per_day #4hr or 12hr shift
    self.job_role = job_role #Supervisor or FOH Assistant

  @property
  def remaining(self) -> float:
    return self.contract_hours - self.assigned_hours

@dataclass(frozen=True)
class Shift:
  day: date
  start: time
  end: time
  hours: float
  required: bool = True


In [19]:
def make_hosp_shifts(start_day: date) -> list[Shift]:
  hosp_shifts = []
  for i in range(7):
    d = start_day + timedelta(days=i)
    hosp_shifts.append(Shift(d, time(8, 0), time(16, 0), 7.5, True))
    hosp_shifts.append(Shift(d, time(15, 0), time(23, 0), 7.5, True))
    hosp_shifts.append(Shift(d, time(11, 0), time(19, 0), 7.5, False))
  return hosp_shifts

def make_cinema_shifts(start_day: date) -> list[Shift]:
  cinema_shifts = []
  for i in range(7):
    d = start_day + timedelta(days=i)
    cinema_shifts.append(Shift(d, time(9, 0), time(17, 0), 8.0, True))
    cinema_shifts.append(Shift(d, time(15, 0), time(23, 0), 8.0, True))
  return cinema_shifts

In [20]:
def build_rota(employees: list[Employee], shifts: list[Shift]):
  rota = []
  shifts_assigned = {e.id: 0 for e in employees}
  worked_days = {e.id: set() for e in employees}
  last_end_dt = {e.id: None for e in employees}

  def shift_datetimes(shift: Shift) -> bool:
    start_dt = datetime.combine(shift.day, shift.start)
    end_dt = datetime.combine(shift.day, shift.end)
    return start_dt, end_dt

  def can_take_shift(e: Employee, shift: Shift) -> bool:
    if shifts_assigned[e.id] >= e.max_shifts_per_week:
      return False
    if shift in worked_days[e.id]:
      return False
    prev_end = last_end_dt[e.id]
    if prev_end is not None:
      start_dt, _ = shift_datetimes(shift)
      if start_dt - prev_end < timedelta(hours=12):
        return False
    return True

  def assign(shift: Shift, chosen: Employee):
    _, end_dt = shift_datetimes(shift)
    chosen.assigned_hours += shift.hours
    shifts_assigned[chosen.id] += 1
    worked_days[chosen.id].add(shift.day)
    last_end_dt[chosen.id] = end_dt
    rota.append((shift, chosen.name))

  required_shifts = [s for s in shifts if getattr(s, "required", True)]
  optional_shifts = [s for s in shifts if not getattr(s, "required", True)]

  for shift in required_shifts:
    eligible = [e for e in employees if shifts_assigned[e.id] < e.max_shifts_per_week]
    if not eligible:
      raise ValueError(f"No elligible employees left for shift {shift} (Everybody is working their max shifts)")

    def score(e: Employee):
      before = e.assigned_hours
      after = before + shift.hours
      below_min_before = before < e.contract_hours
      gap_before = e.contract_hours - before
      closeness_after = -abs(e.contract_hours - after)

      return(below_min_before, gap_before, closeness_after, -shifts_assigned[e.id])

    chosen = max(eligible, key=score)
    chosen.assigned_hours += shift.hours
    shifts_assigned[chosen.id] += 1
    rota.append((shift, chosen.name))

  return rota


In [22]:
hosp_supervisors = [
    Employee("Aleks", 37.5, 5, 8, "Supervisor")
]

cinema_supervisors = [
    Employee("George", 21, 3, 8, "Supervisor")
]

foh_supervisors = [
    Employee("Ben", 21, 3, 8, "Supervisor"),
    Employee("V", 21, 3, 8, "Supervisor"),
    Employee("Kyle", 21, 3, 8, "Supervisor"),
    Employee("Millie", 21, 3, 8, "Supervisor"),
    Employee("Luke C", 21, 3, 8, "Supervisor")
]

team = foh_supervisors + hosp_supervisors
shifts = make_hosp_shifts(date(2026, 3, 2))
hosp_rota = build_rota(team, shifts)

for shift, name in hosp_rota[:14]:
  print(shift.day, shift.start, shift.end, shift.hours, "->", name)

print("\nTotals:")
for e in team:
  print(e.name, e.assigned_hours, "/", e.contract_hours)

2026-03-02 08:00:00 16:00:00 8.0 -> Aleks
2026-03-02 15:00:00 23:00:00 8.0 -> Aleks
2026-03-03 08:00:00 16:00:00 8.0 -> Aleks
2026-03-03 15:00:00 23:00:00 8.0 -> Ben
2026-03-04 08:00:00 16:00:00 8.0 -> V
2026-03-04 15:00:00 23:00:00 8.0 -> Kyle
2026-03-05 08:00:00 16:00:00 8.0 -> Millie
2026-03-05 15:00:00 23:00:00 8.0 -> Luke C
2026-03-06 08:00:00 16:00:00 8.0 -> Aleks
2026-03-06 15:00:00 23:00:00 8.0 -> Ben
2026-03-07 08:00:00 16:00:00 8.0 -> V
2026-03-07 15:00:00 23:00:00 8.0 -> Kyle
2026-03-08 08:00:00 16:00:00 8.0 -> Millie
2026-03-08 15:00:00 23:00:00 8.0 -> Luke C

Totals:
Ben 16.0 / 21
V 16.0 / 21
Kyle 16.0 / 21
Millie 16.0 / 21
Luke C 16.0 / 21
Aleks 32.0 / 37.5
